In [1]:
# Sentiment Analysis with LSTM using PyTorch

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

# Load the dataset (replace 'path/to/dataset.csv' with your file path)
df = pd.read_csv('financial_dataset.csv')

# Assuming the dataset has columns 'text' and 'sentiment'
X = df['text'].astype(str)
y = df['sentiment']

# Encode labels
y = LabelEncoder().fit_transform(y)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Tokenization
tokenizer = get_tokenizer("basic_english")
vocab = build_vocab_from_iterator(map(tokenizer, X_train), specials=["<unk>"])
vocab.set_default_index(vocab["<unk>"])

# Custom dataset class
class SentimentDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = tokenizer(self.texts[idx])
        indices = [vocab[token] for token in tokens]
        return torch.tensor(indices), torch.tensor(self.labels[idx])

# Create datasets and dataloaders
train_dataset = SentimentDataset(X_train.tolist(), y_train)
test_dataset = SentimentDataset(X_test.tolist(), y_test)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

# LSTM model definition
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(LSTMModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)
        output = self.fc(lstm_out[:, -1, :])
        return torch.sigmoid(output)

# Instantiate model, loss function, and optimizer
model = LSTMModel(len(vocab), 128, 128)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
for epoch in range(5):
    model.train()
    for texts, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(texts).squeeze()
        loss = criterion(outputs, labels.float())
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# Evaluation
model.eval()
correct = 0
with torch.no_grad():
    for texts, labels in test_loader:
        outputs = model(texts).squeeze()
        predicted = (outputs > 0.5).float()
        correct += (predicted == labels).sum().item()

accuracy = correct / len(test_dataset)
print(f"Accuracy: {accuracy * 100:.2f}%")

# Prediction with a new text
new_text = ["Coca-Cola has risen with 200M of benefits"]
tokens = tokenizer(new_text[0])
indices = torch.tensor([vocab[token] for token in tokens]).unsqueeze(0)
model.eval()
prediction = model(indices).item()
sentiment = 'positive' if prediction > 0.5 else 'negative'
print(f'Predicted sentiment: {sentiment}')

OSError: [WinError 127] No se encontró el proceso especificado